# Arm SDK low-level joint control

This notebook creates a Jupyter joint-control UI for `rt/arm_sdk`. Participants select an arm joint, nudge or set its target pose, and the publisher ramps every command incrementally at a configured speed.

`rt/arm_sdk` controls the upper body only and does not require developer mode, but it still sends real servo targets. Sync to the live state before moving.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import DDS, widgets, and the same arm joint constants used by the local slider app.


In [ ]:
import threading
import time
from dataclasses import dataclass
from typing import Any

import ipywidgets as widgets
from IPython.display import display

from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

LEFT_ARM_IDX = [15, 16, 17, 18, 19, 20, 21]
RIGHT_ARM_IDX = [22, 23, 24, 25, 26, 27, 28]
NOT_USED_IDX = 29
JOINT_NAMES = ["shoulder_pitch", "shoulder_roll", "shoulder_yaw", "elbow", "wrist_pitch", "wrist_roll", "wrist_yaw"]
ALL_ARM_JOINTS = LEFT_ARM_IDX + RIGHT_ARM_IDX

from sdk_client import Robot
from inspire_sdk import close_hand as inspire_close_hand, open_hand as inspire_open_hand


Define a lowstate reader and a ramping publisher. The publish loop limits joint target changes by `speed_rad_s / rate_hz` on every tick.


In [ ]:
def resolve_lowstate_type():
    # TODO: Try the supported Unitree DDS modules and return the first LowState_ type that imports successfully.
    raise NotImplementedError("Participant exercise: complete resolve_lowstate_type.")


class ArmState:
    def __init__(self, joints):
        # TODO: Initialize instance fields, clients, publishers/subscribers, locks, and default state needed by the class.
        raise NotImplementedError("Participant exercise: complete __init__.")

    def _callback(self, msg: Any):
        # TODO: Decode incoming SDK/DDS messages and store the latest relevant state safely.
        raise NotImplementedError("Participant exercise: complete _callback.")

    def snapshot(self):
        # TODO: Return a thread-safe copy of the latest cached state.
        raise NotImplementedError("Participant exercise: complete snapshot.")

    def wait(self, timeout=3.0):
        # TODO: Poll until fresh data is available or raise a timeout.
        raise NotImplementedError("Participant exercise: complete wait.")


class ArmRampController:
    def __init__(self, iface, domain_id, joints):
        # TODO: Initialize instance fields, clients, publishers/subscribers, locks, and default state needed by the class.
        raise NotImplementedError("Participant exercise: complete __init__.")

    def sync(self):
        # TODO: Read live robot state and use it as the current command target.
        raise NotImplementedError("Participant exercise: complete sync.")

    def set_target(self, joint, q):
        # TODO: Store a new desired joint target from the UI.
        raise NotImplementedError("Participant exercise: complete set_target.")

    def nudge(self, joint, delta):
        # TODO: Offset the selected joint target by the requested delta.
        raise NotImplementedError("Participant exercise: complete nudge.")

    def start(self):
        # TODO: Start background publishing/control if it is not already running.
        raise NotImplementedError("Participant exercise: complete start.")

    def stop(self):
        # TODO: Request the background activity to stop and leave the robot in a safe state.
        raise NotImplementedError("Participant exercise: complete stop.")

    def zero_gains_once(self):
        # TODO: Publish one command with zero gains to release active stiffness.
        raise NotImplementedError("Participant exercise: complete zero_gains_once.")

    def _loop(self):
        # TODO: Run the timed background control loop until stop is requested.
        raise NotImplementedError("Participant exercise: complete _loop.")

    def _publish_locked(self):
        # TODO: Fill the SDK command message from current targets/gains and publish it with CRC.
        raise NotImplementedError("Participant exercise: complete _publish_locked.")

arm_control = ArmRampController(IFACE, DOMAIN_ID, ALL_ARM_JOINTS)
print("Arm ramp controller ready.")


robot_control = None


def get_robot_control():
    # TODO: Lazily create and cache the high-level Robot control client.
    raise NotImplementedError("Participant exercise: complete get_robot_control.")


def normalize_hand_selection(hand):
    # TODO: Map the UI hand selector into the sides expected by the hand SDK.
    raise NotImplementedError("Participant exercise: complete normalize_hand_selection.")


def hand_action(hand_type, action, hand="both", hold_s=0.6, ramp_s=0.4):
    # TODO: Dispatch open/close commands to the selected hand implementation and return a status dict.
    raise NotImplementedError("Participant exercise: complete hand_action.")


Run the UI. Select one joint, sync to the live state, start the ramping publisher, and then make small changes.


In [ ]:
options = []
for side, joints in (("left", LEFT_ARM_IDX), ("right", RIGHT_ARM_IDX)):
    for offset, joint in enumerate(joints):
        options.append((f"{side} {JOINT_NAMES[offset]} ({joint})", joint))
joint = widgets.Dropdown(options=options, description="Joint")
target = widgets.FloatSlider(value=0.0, min=-3.14, max=3.14, step=0.01, description="Target rad", readout_format=".2f", layout=widgets.Layout(width="520px"))
nudge = widgets.FloatSlider(value=0.05, min=0.005, max=0.25, step=0.005, description="Nudge")
speed = widgets.FloatSlider(value=0.35, min=0.02, max=1.5, step=0.02, description="Speed")
kp = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0, description="kp")
kd = widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description="kd")
hand_type = widgets.Dropdown(options=["dex3", "inspire"], value="dex3", description="Hand type")
hand_side = widgets.Dropdown(options=["both", "left", "right"], value="both", description="Hand")
sync = widgets.Button(description="Sync", button_style="info")
start = widgets.Button(description="Start Ramp", button_style="success")
stop = widgets.Button(description="Pause", button_style="warning")
minus = widgets.Button(description="- Nudge")
plus = widgets.Button(description="+ Nudge")
zero = widgets.Button(description="Zero Gains", button_style="danger")
release = widgets.Button(description="Release Arms", button_style="warning")
reengage = widgets.Button(description="Reengage Arms", button_style="success")
open_hand_btn = widgets.Button(description="Open Hand")
close_hand_btn = widgets.Button(description="Close Hand")
status = widgets.HTML(value="")


def set_status(value):
    # TODO: Serialize status payloads into the notebook text area.
    raise NotImplementedError("Participant exercise: complete set_status.")


def refresh_target_from_desired(*_):
    # TODO: Complete the implementation for refresh_target_from_desired using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete refresh_target_from_desired.")


def apply_gains(*_):
    # TODO: Complete the implementation for apply_gains using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete apply_gains.")


def on_target(change):
    # TODO: Complete the implementation for on_target using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_target.")


def on_release(_):
    # TODO: Complete the implementation for on_release using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_release.")


def on_reengage(_):
    # TODO: Complete the implementation for on_reengage using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_reengage.")


def on_hand(action):
    # TODO: Complete the implementation for on_hand using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_hand.")

sync.on_click(lambda _: (setattr(status, "value", arm_control.sync()), refresh_target_from_desired()))
start.on_click(lambda _: (apply_gains(), setattr(status, "value", arm_control.start())))
stop.on_click(lambda _: setattr(status, "value", arm_control.stop()))
minus.on_click(lambda _: (setattr(status, "value", arm_control.nudge(joint.value, -nudge.value)), refresh_target_from_desired()))
plus.on_click(lambda _: (setattr(status, "value", arm_control.nudge(joint.value, nudge.value)), refresh_target_from_desired()))
zero.on_click(lambda _: setattr(status, "value", arm_control.zero_gains_once()))
release.on_click(on_release)
reengage.on_click(on_reengage)
open_hand_btn.on_click(lambda _: on_hand("open"))
close_hand_btn.on_click(lambda _: on_hand("close"))
joint.observe(lambda change: refresh_target_from_desired(), names="value")
target.observe(on_target, names="value")
for w in (speed, kp, kd):
    w.observe(lambda change: apply_gains(), names="value")
refresh_target_from_desired()
display(widgets.VBox([
    widgets.HBox([joint, target]),
    widgets.HBox([nudge, speed]),
    widgets.HBox([kp, kd]),
    widgets.HBox([sync, start, stop, minus, plus, zero]),
    widgets.HBox([release, reengage]),
    widgets.HBox([hand_type, hand_side, open_hand_btn, close_hand_btn]),
    status,
]))
